# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a demonstration for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata as Python object
metadata = dataset.metadata

# Pretty print title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s. All references are by their `@id` fields following best practice.

In [ ]:
# List all available record sets
print('Available Record Sets:')
record_sets = []
for record_set in dataset.record_sets:
    print(f"- Name: {record_set.name}, @id: {record_set.id}")
    record_sets.append(record_set.id)
    # List fields for the record set
    print('  Fields:')
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}) | dataType: {getattr(field, 'data_type', None)}")

### Preview some records from each record set
Each record is returned as a dictionary; fields are referenced by their canonical `@id`s.

In [ ]:
# Display a few records from each available record set
for record_set_id in record_sets:
    print(f"\nRecords for Record Set @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        pprint.pprint(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s found in the overview above.

In [ ]:
# Collect all record sets' data into DataFrames, indexed by record set @id
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for Record Set @id {record_set_id} has shape {df.shape}")
    print(f"Columns: {list(df.columns)}")

## 4. Exploratory Data Analysis (EDA)
For demonstration, we'll operate on a record set with numeric fields. The exact field and record set `@id`s are determined above. We will:
- Filter records by a numeric threshold.
- Normalize a numeric column.
- Group (aggregate) by a relevant field.

In [ ]:
# For illustration, select the first record set that has a numeric field
target_record_set_id = None
numeric_field_id = None
group_field_id = None

# Identify a good numeric field for demonstration
for record_set in dataset.record_sets:
    df = dataframes[record_set.id]
    for field in record_set.fields:
        # Try to pick first numeric-looking field (float or int columns)
        if getattr(field, 'data_type', None) in ['Float', 'Integer', 'Number']:
            if field.id in df.columns:
                target_record_set_id = record_set.id
                numeric_field_id = field.id
                # Try to pick a group field (categorical, not this one)
                for group_field_candidate in record_set.fields:
                    if getattr(group_field_candidate, 'data_type', '') == 'Text' and group_field_candidate.id in df.columns:
                        group_field_id = group_field_candidate.id
                        break
                break
    if target_record_set_id:
        break

print(f"Using record set: {target_record_set_id}")
print(f"Numeric field for demo: {numeric_field_id}")
if group_field_id:
    print(f"Group (categorical) field: {group_field_id}")

# Proceed if we found a numeric field
if target_record_set_id and numeric_field_id:
    df = dataframes[target_record_set_id].copy()
    # Convert column to numeric (just in case)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75) if df[numeric_field_id].notna().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std if std > 0 else 0
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouped means by group field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the numeric field of interest and its distribution, and show aggregation/grouping if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id and numeric_field_id:
    df = dataframes[target_record_set_id].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We used the FAIR² dataset Croissant schema to programmatically discover the dataset's structure and load its records by referencing all entities by their canonical `@id`. 
We demonstrated extraction, filtering, normalization, and visualization for fields, making it easy to extend the workflow for downstream research and reproducible clinical analytics.